In [ ]:
%pip install lxml bs4 langchain langchain-openai faiss-cpu networkx matplotlib
dbutils.library.restartPython()

In [ ]:
import os
import re
import uuid
import heapq
import networkx as nx
import matplotlib.pyplot as plt

from bs4 import BeautifulSoup
from pyspark.sql import Row
from pyspark.sql.functions import monotonically_increasing_id

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain.vectorstores import FAISS
from langchain.schema import Document

In [ ]:
def clean_xwiki_text(text):
    text = re.sub(r"<!\[CDATA\[|\]\]>", "", text)
    text = BeautifulSoup(text, "html.parser").get_text(" ")
    text = re.sub(r"\[\[.*?\]\]|\{\{.*?\}\}", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
def split_chunks(text, metadata):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200
    )

    chunks = splitter.split_text(text)

    return [
        {
            "text": c,
            "metadata": {**metadata, "chunk_index": i}
        }
        for i, c in enumerate(chunks)
    ]

In [ ]:
xml_paths = [
    "/Volumes/...3829c0e9.../WebHome.xml",
    "/Volumes/...76f049ea.../WebHome.xml",
    "/Volumes/...07730902.../WebHome.xml",
    "/Volumes/...f2280f6d.../WebHome.xml",
    "/Volumes/...96bf76a9.../WebHome.xml"
]

rows = []

for path in xml_paths:
    with open(path, "rb") as f:
        soup = BeautifulSoup(f.read(), "xml")

    content = soup.find("content").text if soup.find("content") else ""
    title = soup.find("title").text if soup.find("title") else ""

    xwikidoc = soup.find("xwikidoc")
    reference = xwikidoc.get("reference") if xwikidoc else None

    rows.append(Row(
        content=content,
        source=path,
        title=title,
        reference=reference
    ))

df_raw = spark.createDataFrame(rows)
df_raw = df_raw.withColumn("doc_id", monotonically_increasing_id())

display(df_raw)

In [ ]:
chunk_rows = []

for row in df_raw.collect():
    cleaned = clean_xwiki_text(row["content"])

    chunks = split_chunks(cleaned, {"doc_id": str(row["doc_id"])})

    for c in chunks:
        chunk_rows.append(Row(
            id=str(uuid.uuid4()),
            doc_id=str(row["doc_id"]),
            content=c["text"]
        ))

df_chunks = spark.createDataFrame(chunk_rows)

display(df_chunks)
print("Total chunks:", df_chunks.count())

In [ ]:
G = nx.DiGraph()

rows = df_raw.collect()

for r in rows:
    G.add_node(str(r["doc_id"]), content=r["content"])

for r in rows:
    if r["reference"]:
        G.add_edge(str(r["doc_id"]), str(r["reference"]))

pagerank_scores = nx.pagerank(G)

for node in G.nodes():
    G.nodes[node]["pagerank"] = pagerank_scores.get(node, 0)

print("Nodes:", len(G.nodes()))
print("Edges:", len(G.edges()))

In [ ]:
embedding_model = OpenAIEmbeddings()

docs = []

for r in df_chunks.collect():
    docs.append(
        Document(
            page_content=r["content"],
            metadata={"doc_id": str(r["doc_id"])}
        )
    )

vector_store = FAISS.from_documents(docs, embedding_model)

print("Vector store ready")

In [ ]:
def graph_retriever(graph, vector_store, query, max_hops=2, top_k=10):

    retrieved_docs = vector_store.similarity_search(query, k=top_k)

    start_nodes = set()
    for d in retrieved_docs:
        if "doc_id" in d.metadata:
            start_nodes.add(d.metadata["doc_id"])

    print("Start Nodes:", start_nodes)

    visited = set()
    scores = {}
    queue = []
    traversal_path = []

    for node in start_nodes:
        pr = graph.nodes[node].get("pagerank", 0)
        scores[node] = pr
        queue.append((node, 0))

    while queue:
        current, hop = queue.pop(0)

        if hop >= max_hops or current in visited:
            continue

        visited.add(current)
        traversal_path.append(current)

        print("Visited:", current)

        for neighbor in graph.neighbors(current):
            pr = graph.nodes[neighbor].get("pagerank", 0)
            score = pr * (1 / (hop + 1))

            if neighbor not in scores or score > scores[neighbor]:
                scores[neighbor] = score

            queue.append((neighbor, hop + 1))

    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    top_nodes = [node for node, _ in ranked[:top_k]]

    print("Top Nodes:", top_nodes)

    return top_nodes, traversal_path

In [ ]:
def visualize_graph(graph, path=None):

    plt.figure(figsize=(12, 8))
    pos = nx.spring_layout(graph, seed=42)

    nx.draw(graph, pos, with_labels=True,
            node_color="lightblue", node_size=2000)

    if path and len(path) > 1:
        edges = list(zip(path, path[1:]))

        nx.draw_networkx_edges(
            graph, pos,
            edgelist=edges,
            edge_color="red",
            width=3
        )

        nx.draw_networkx_nodes(
            graph, pos,
            nodelist=path,
            node_color="orange"
        )

    plt.title("Graph (Red = Traversal Path)")
    plt.show()

In [ ]:
class FinalGraphRAG:

    def __init__(self, graph, vector_store, llm):
        self.graph = graph
        self.vector_store = vector_store
        self.llm = llm

    def answer(self, query):

        nodes, path = graph_retriever(
            self.graph,
            self.vector_store,
            query
        )

        visualize_graph(self.graph, path)

        context = ""
        for n in nodes:
            context += "\n" + self.graph.nodes[n]["content"]

        prompt = f"""
        Answer using context:

        {context}

        Question: {query}
        """

        return self.llm.invoke(prompt)

In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

rag = FinalGraphRAG(G, vector_store, llm)

response = rag.answer("Explain your topic")

print(response)